In [2]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

In [3]:
from decimal import Decimal

data = [
    [1, "Ivanov","Ivan", None],
    [2, "Petrov","Petya", Decimal('100')],
    [3, None, "Vasya", Decimal('200')],
    [4, None, None, None],
]
schema = "id Integer, second_name String, first_name String, salary Decimal"

df = spark.createDataFrame(data, schema)
display(df)

DataFrame[id: int, second_name: string, first_name: string, salary: decimal(10,0)]

In [4]:
from pyspark.sql.functions import col, coalesce, lit

# Если second_name null, подставится строка first_name
df01 = df.select(
    "id",
    coalesce(col("second_name"), col("first_name")).alias("name"),
    coalesce("salary", lit(0)).alias("salary"),
)
display(df01)

DataFrame[id: int, name: string, salary: decimal(10,0)]

In [5]:
from pyspark.sql.functions import col, coalesce, lit

# Если имя None, подставится строка "Неизвестно"
df01 = df.select(
    "id",
    coalesce(col("name"), lit("Неизвестно")).alias("name"),
    coalesce(col("salary"), lit(0)).alias("salary")
)
display(df01)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `name` cannot be resolved. Did you mean one of the following? [`id`, `salary`, `first_name`, `second_name`]. SQLSTATE: 42703;
'Project [id#57602, 'coalesce('name, Неизвестно) AS name#57610, coalesce(salary#57605, cast(0 as decimal(10,0))) AS salary#57611]
+- LocalRelation [id#57602, second_name#57603, first_name#57604, salary#57605]


In [6]:
df.createOrReplaceTempView("tmp_df")

In [8]:
spark.sql("""
select
    second_name,
    ifnull(second_name, "Unknown") as ifnull_name,
    nvl(second_name, "Unknown") as nvl_name,
    nvl2(second_name, second_name, 'Unknown') as nvl2_name,
    nullif(second_name, "Petrov") as nullif_name
from tmp_df
""")

DataFrame[second_name: string, ifnull_name: string, nvl_name: string, nvl2_name: string, nullif_name: string]

In [9]:
from pyspark.sql.functions import ifnull, nvl, nvl2, nullif, col, lit

df_tmp = df.select(
    "second_name",
    ifnull(col("second_name"), lit("Unknown")).alias("ifnull_name"),
    nvl(col("second_name"), lit("Unknown")).alias("nvl_name"),
    nvl2(col("second_name"), col("second_name"), lit("Unknown")).alias("nvl2_name"),
    nullif(col("second_name"), lit("Petrov")).alias("nullif_name")
)
display(df_tmp)

DataFrame[second_name: string, ifnull_name: string, nvl_name: string, nvl2_name: string, nullif_name: string]

In [10]:
df.na.drop().show()
df.na.drop("any").show()
df.na.drop("all", subset=['second_name', 'first_name', 'salary']).show()
df.na.drop("any", subset=['second_name']).show()

+---+-----------+----------+------+
| id|second_name|first_name|salary|
+---+-----------+----------+------+
|  2|     Petrov|     Petya|   100|
+---+-----------+----------+------+

+---+-----------+----------+------+
| id|second_name|first_name|salary|
+---+-----------+----------+------+
|  2|     Petrov|     Petya|   100|
+---+-----------+----------+------+

+---+-----------+----------+------+
| id|second_name|first_name|salary|
+---+-----------+----------+------+
|  1|     Ivanov|      Ivan|  NULL|
|  2|     Petrov|     Petya|   100|
|  3|       NULL|     Vasya|   200|
+---+-----------+----------+------+

+---+-----------+----------+------+
| id|second_name|first_name|salary|
+---+-----------+----------+------+
|  1|     Ivanov|      Ivan|  NULL|
|  2|     Petrov|     Petya|   100|
+---+-----------+----------+------+



In [11]:
from pyspark.sql import Row
df = spark.createDataFrame([
    Row(age=10, height=80.0, name="Alice"),
    Row(age=5, height=float("nan"), name="Bob"),
    Row(age=None, height=None, name="Tom"),
    Row(age=None, height=float("nan"), name=None),
])



In [12]:
#DataFrame.dropna(how='any', thresh=None, subset=None)

print("Example 1: Drop the row if it contains any null or NaN.")
df.na.drop().show()

print("Example 2: Drop the row only if all its values are null or NaN.")
df.na.drop(how = "all").show()

print("Example 3: Drop rows that have less than thresh non-null and non-NaN values.")
# .drop(thresh=2) — указывает Spark оставить строку только в том случае, если в ней есть как минимум 2 заполненных столбца. Если в 
# строке только один заполненный столбец (или все пустые), она будет удалена.
# thresh имеет выше приоритет, чем how
df.na.drop(thresh=2).show()

print("Example 4: Drop rows with null and NaN values in the specified columns.")
df.na.drop(subset=['age', 'name']).show()


Example 1: Drop the row if it contains any null or NaN.
+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
+---+------+-----+

Example 2: Drop the row only if all its values are null or NaN.
+----+------+-----+
| age|height| name|
+----+------+-----+
|  10|  80.0|Alice|
|   5|   NaN|  Bob|
|NULL|  NULL|  Tom|
+----+------+-----+

Example 3: Drop rows that have less than thresh non-null and non-NaN values.
+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
|  5|   NaN|  Bob|
+---+------+-----+

Example 4: Drop rows with null and NaN values in the specified columns.
+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
|  5|   NaN|  Bob|
+---+------+-----+



In [15]:
df_tmp3 = df.na.fill("Unknown").na.fill(0)
df_tmp3.show()
df_tmp3 = df.na.fill({"second_name": "****", "salary": 0})
df_tmp3.show()
df_tmp3 = df.na.replace(["Petrov"], ["Pupkin"], "second_name")
df_tmp3.show()


+---+------+-------+
|age|height|   name|
+---+------+-------+
| 10|  80.0|  Alice|
|  5|   0.0|    Bob|
|  0|   0.0|    Tom|
|  0|   0.0|Unknown|
+---+------+-------+



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `second_name` cannot be resolved. Did you mean one of the following? [`age`, `height`, `name`]. SQLSTATE: 42703